In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import seaborn as sns
import numpy as np
import munch
import yaml 
import math
from matplotlib.colors import LogNorm

#from highres_ta.scripts import utils

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
plt.style.use('default')

    # drop maxsampdepth

    # qc, flag, error, uncert

# Utils

In [ ]:
def read_yaml(fname: str)-> munch.Munch:


    with open(fname) as fobj:
        config = yaml.safe_load(fobj)
        
    #config = munch.munchify(config)

    return config

In [ ]:
def spatial_bins(df, lon_mesh=0.25, lat_mesh=0.25, histogram = False):
    
    bins_lon = np.arange(-180, 181, lon_mesh, dtype=float).tolist()
    bins_lat = np.arange(-90, 91, lat_mesh, dtype=float).tolist()

    labels_lon = np.convolve(np.array(bins_lon), [0.5, 0.5], mode='valid').tolist()
    labels_lat = np.convolve(np.array(bins_lat), [0.5, 0.5], mode='valid').tolist()

    lat_bin_serie = pd.cut(df['lat_gp'], bins_lat, labels=labels_lat).values.astype(float)
    lon_bin_serie = pd.cut(df['lon_gp'], bins_lon, labels=labels_lon).values.astype(float)
    
    
    if histogram == True:
        fig, axes = plt.subplots(2,1, figsize = (20,10))
        
        sns.histplot(lat_bin_serie, discrete = True, ax = axes[1])
        axes[1].set_xlabel('lat')
        axes[1].text(0.05, 0.95, f"Bin mesh {lat_mesh}", transform=axes[1].transAxes,verticalalignment='top',bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        sns.histplot(lon_bin_serie, discrete = True, ax = axes[0])
        axes[0].set_xlabel('lon')
        axes[0].text(0.05, 0.95, f"Bin mesh {lon_mesh}", transform=axes[0].transAxes,verticalalignment='top',bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        plt.show();
    
    return lon_bin_serie, lat_bin_serie
    

def quantile_bins(df, variable, quantiles = (np.array([0, 1, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 99, 100]) / 100), plot = False):

    binned_var, var_quantiles  = pd.qcut(df['variable'], q = quantiles, retbins=True)
    midpoints = binned_var.apply(lambda x: x.mid).astype(float)
    
    if plot == True:

        sns.countplot(x=binned_var, order=binned_var.cat.categories)
        plt.xticks(rotation=90)
        plt.show();

        counts = binned_var.value_counts().sort_index()

        plt.bar(
            salinity_var[:-1],
            counts,
            width=np.diff(var_quantiles),
            align='edge',
            edgecolor = 'black'
        )
        plt.xlabel(f"{variable}")
        plt.ylabel("Count")
        plt.show();

    # midpoints = np.convolve(bins, [0.5, 0.5], mode='valid').tolist()
    # sns.histplot(binned_sal)
    # plt.show()

    return binned_sal, midpoints


# def salinity_bins(df, quantiles=np.array([0,1,5,10,20,30,40,50,60,70,80,90,95,99,100])/100):

#     binned_sal, bins = pd.qcut(
#         df['salinity_gp'],
#         q=quantiles,
#         retbins=True
#     )

#     df['sal_bin'] = binned_sal
#     df['sal_bin_mid'] = binned_sal.apply(lambda x: x.mid)

#     return df, bins

In [ ]:
def latlon_histplot(lon_serie, lat_serie, title):
    
    fig, axes = plt.subplots(2,1, figsize = (20,10))
            
    sns.histplot(lat_serie, discrete = True, ax = axes[1])
    axes[1].set_xlabel('lat')
    
    sns.histplot(lon_serie, discrete = True, ax = axes[0])
    axes[0].set_xlabel('lon')
    
    fig.suptitle(title)
    plt.tight_layout()
    plt.show();

In [ ]:
def blank_map(projection = ccrs.PlateCarree(), lon_res = 1.0, lat_res = 1.0 ):

    # Set grid resolution (change this)

    fig = plt.figure(figsize=(12, 10))
    ax = plt.axes(projection=projection)

    ax.set_global()
    ax.coastlines()

    # Define tick locations
    lon_ticks = np.arange(-180, 181, lon_res)
    lat_ticks = np.arange(-90, 91, lat_res)

    gl = ax.gridlines(
        crs=projection,
        draw_labels=False,
        linewidth=0.5,
        color='gray',
        alpha=0.5,
        linestyle='--'
    )

    gl.xlocator = plt.FixedLocator(lon_ticks)
    gl.ylocator = plt.FixedLocator(lat_ticks)


    ax.add_feature(cfeature.LAND, zorder = 10, facecolor='white')
    ax.add_feature(cfeature.OCEAN, facecolor='white')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)

    
    return fig, ax

def samples_map(df:pd.DataFrame, source_dataset='gp', lon_res = 1.0, lat_res = 1.0, color = 'blue'):
    
    lon_str = f"lon_{source_dataset}"
    lat_str = f"lat_{source_dataset}"
    
    fig, ax = blank_map(lon_res=lon_res, lat_res=lat_res)
    
    ax.scatter(
        df[lon_str],
        df[lat_str],
        marker = 'x',
        s = 10,
        color = color,
        transform = ccrs.PlateCarree()
    )
    plt.show();
    
    
def blank_subplots(n_plots, n_cols):
    
    """Grid of subplots spread on n_cols and with n_rows tailored to total number of variables to plot n_plots"""
    
    # Compute number of rows and columns
    n_rows = math.ceil(n_plots / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows), sharex=False, sharey=True)
    axes = axes.flatten()  # flatten to index easily
    plt.grid()
    
    return fig, axes

def remaining_axes_del(fig, axes, n_plots, n_cols):
    
    n_rows = math.ceil(n_plots / n_cols)
    
    for j in range(n_plots, n_rows*n_cols):
        fig.delaxes(axes[j])

# Data imports

In [ ]:

# ONLINE

url = "https://data.up.ethz.ch/shared/OceanSODA-ETHZv2/total_alkalinity/GLODAPv2023/GLODAPv2023-raw_collocated-{y}.pq"
df = pd.concat([pd.read_parquet(url.format(y=y)) for y in range(1982, 2022)])


# # Save on local file (working local)
# df.to_csv(f"matched_GLODAP2023.csv", index=True)

# New names convention
renaming_dict = read_yaml('/home/edupuis/highres_TA/renaming_dict.yaml')
df = (df.drop(columns='maxsampdepth')
        .rename(columns = renaming_dict)
)


# # Monthly/8-daily bins and time column reorganizing
# col = df.pop('year_gp')
# df.insert(2, 'year_gp', col)
# df.insert(3, 'month_gp', df['time_gp'].dt.month)

# # 8Dx0.25x0.25 bins 
# df.insert(4, 'bins_8D_gp', ((df['time_gp'].dt.dayofyear - 1) // 8 ).astype(int))
# lon_bins_serie, lat_bin_serie = spatial_bins(df, 0.25, 0.25)
# df.insert(8, 'bins_025lat_gp', lat_bin_serie)
# df.insert(9, 'bins_025lon_gp', lon_bins_serie)

# # move alk column to the front
# col = df.pop('talk_gp')
# df.insert(0, 'talk_gp', col)

# store column names       
all_cols = df.columns.to_list()
all_time_cols = [i for i in all_cols if 'time' in i]
all_coord_cols = [i for i in all_cols if ('lat' in i or 'lon' in i or 'time' in i)]
bins_col = [i for i in all_cols if 'bins' in i]


# print info
df.info()

In [ ]:
# OFFLINE

# import from local file
df = pd.read_csv(f"matched_GLODAP2023.csv")

#Naming convention and drop sampledepth and index column 
renaming_dict = read_yaml('C:/Users/elsad/Git/TA_highres/highres_TA/renaming_dict.yaml')
df = df.drop(columns=['maxsampdepth', df.columns[0]]).rename(columns = renaming_dict)

# column names
all_cols = df.columns.to_list()
all_time_cols = [i for i in all_cols if 'time' in i]
all_coord_cols = [i for i in all_cols if ('lat' in i or 'lon' in i or 'time' in i)]

# reattribute correct datetime types
df = (df.astype({col : 'datetime64[ns]' for col in all_time_cols})
      .astype({'expocode_gp': 'string'})
)

# move alk column to the front
col = df.pop('talk_gp')
df.insert(0, 'talk_gp', col)

# cols list update
all_cols = df.columns.to_list()

# print_info
df.info()

# distribution map
samples_map(df)

In [ ]:
# Monthly/8-daily bins and time column reorganizing
col = df.pop('year_gp')
df['bins_year_gp'] = col
df['bins_month_gp'] = df['time_gp'].dt.month

# 8Dx0.25x0.25 bins 
df['bins_8D_gp'] = (((df['time_gp'].dt.dayofyear - 1) // 8 ).astype(int))
lon_bins_serie, lat_bin_serie = spatial_bins(df, 0.25, 0.25)
df['bins_025lat_gp'] = lat_bin_serie
df['bins_025lon_gp'] = lon_bins_serie




bins_cols = [i for i in all_cols if 'bins' in i]

# cols list update
all_cols = df.columns.to_list()

In [ ]:
# per variables categories
salinity_cols = ['salinity_gp','salinity_soda','sss_glorys','sss_multiobs','sss_cciS']
biochem_cols = ['oxygen_gp','aou_gp','nitrate_gp','nitrite_gp','silicate_gp','phosphate_gp','tco2_gp','fco2_gp','phtsinsitutp_gp', 'chl_globcolour']
mld_cols = ['mld_soda','mld_glorys']
temp_cols = ['temp_gp','temp_soda','sst_cciT','ice_cciT']
spatiotemporal_cols = ['time_gp','lat_gp','lon_gp','depth_gp','bottomdepth_gp']
dynamic_cols = ['ssh_adt_cmems','ssh_sla_cmems']
metadata_cols = ['sst_uncert_cciT','sss_error_multiobs','sss_random_error_cciS','depth_multiobs','depth_soda','talk_flag_gp','talk_qc_gp','chl_flags_globcolour','chl_uncert_globcolour']


# per dataset columns

def perdataset_columns(df:pd.DataFrame)-> dict:
    
    """Enter the df you are working with (cleaned or extended) and return dict = {dataset_name : list cols}"""
    
    cnames = df.columns.tolist()
    
    glodap_cols = [i for i in cnames if '_gp' in i]
    soda_cols = [i for i in cnames if '_soda' in i]
    glorys_cols = [i for i in cnames if '_glorys' in i]
    multiobs_cols = [i for i in cnames if '_multiobs' in i]
    globcolour_cols = [i for i in cnames if '_globcolour' in i]
    cmems_cols = [i for i in cnames if '_cmems' in i] 
    cciT_cols = [i for i in cnames if '_cciT' in i] 
    cciS_cols = [i for i in cnames if '_cciS' in i] 
    
    per_dataset_dict = {
        'gp' : glodap_cols,
        'soda' : soda_cols,
        'glorys' : glorys_cols,
        'multiobs': multiobs_cols,
        'globcolour': globcolour_cols,
        'cmems': cmems_cols,
        'cciT': cciT_cols,
        'cciS': cciS_cols
    }
    
    return per_dataset_dict

# Minimalistic df

In [ ]:
# Minimalistic df 

# drop coordinates and metadata
to_drop = [i for i in all_coord_cols if '_gp' not in i]
to_drop = to_drop + metadata_cols

clean_df = df.drop(columns = to_drop)

clean_df.info()

# EDA 

In [ ]:
#pearson correlation (linear relation)

def correlations(df: pd.DataFrame, method: str, heatmap = True):
    
    correlations = df.select_dtypes(include=['number','datetime']).corr(method=method)
    
    if heatmap == True:
        
        cmap_dict = {
            'pearson' : 'coolwarm',
            'spearman': 'viridis',
            'kendall' : 'plasma'
        }
            
        plt.figure(figsize=(20,16))
        sns.heatmap(correlations, annot=True, cmap=cmap_dict[method])
        plt.title(f"{method} correlation heatmap")
        plt.show();
    
    return correlations

def kernel_density_plot(df, variables_list, title = ''):

    #sns.set_style("darkgrid")

    plt.figure(figsize=(14, len(variables_list) * 3))
    for i, feature in enumerate(variables_list, 1):
        plt.subplot(len(variables_list), 2, i)
        sns.histplot(df[feature],kde=True)
        #plt.title(f"{feature}")
        #plt.title(f"{feature} | Skewness: {round(df[feature].skew(), 2)}")

    plt.tight_layout()
    plt.suptitle(title,fontsize=16)
    plt.subplots_adjust(top=0.95)
    plt.show()


def alk_vs_cols(df:pd.DataFrame, variables_list: list, n_cols_plot=2):
    
    # dont plot talk vs talk
    if 'talk_gp' in variables_list:
        variables_list.remove('talk_gp')
    
    #remove non numerical columns
    variables_list = df[variables_list].select_dtypes(include=['number','datetime']).columns.to_list()
    n_plots = len(variables_list)
    
    pearson_corr = correlations(df, method= 'pearson', heatmap=False)
    kendall_corr = correlations(df, method= 'kendall', heatmap=False)   
    
    
    fig, axes = blank_subplots(n_plots = n_plots, n_cols=n_cols_plot)

    for i, col in enumerate(variables_list):
        axes[i].scatter(df[col], df['talk_gp'], alpha=0.5)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('talk_gp')
        

        pcorr =  pearson_corr['talk_gp'].loc[col]  
        taucorr =  kendall_corr['talk_gp'].loc[col]       
        
        axes[i].text(0.05, 0.95, f"r= {pcorr:.2f}, \tau = {taucorr:.2f}", transform=axes[i].transAxes,verticalalignment='top',bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    remaining_axes_del(fig, axes, n_plots,n_cols_plot)
    
    plt.tight_layout()
    plt.show();
    
def variable_vs_bins(df:pd.DataFrame, var:str, n_cols_plot=2):
    
    n_plots = len(bins_cols)
    
    fig, axes = blank_subplots(n_plots = n_plots, n_cols=n_cols_plot)

    for i, col in enumerate(bins_cols):
        axes[i].scatter(df[col],df[var], alpha=0.5)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel(var)       
        

    remaining_axes_del(fig, axes, n_plots,n_cols_plot)
    
    plt.tight_layout()
    plt.show();


In [ ]:
pearson_corr = correlations(clean_df.drop(columns=bins_cols),'pearson')
kendall_corr = correlations(clean_df.drop(columns=bins_cols),'kendall')
spearman_corr = correlations(clean_df.drop(columns=bins_cols),'spearman')

In [ ]:
for var in temp_cols:
    variable_vs_bins(clean_df, var)

In [ ]:
kernel_density_plot(clean_df,bins_cols)
alk_vs_cols(clean_df,bins_cols)

In [ ]:
alk_vs_cols(clean_df, salinity_cols)
alk_vs_cols(clean_df, biochem_cols)
alk_vs_cols(clean_df,spatiotemporal_cols)
alk_vs_cols(clean_df,bins_cols)
alk_vs_cols(clean_df,mld_cols)
alk_vs_cols(clean_df,temp_cols)

In [ ]:
#PER DATASETS
dataset_dict = perdataset_columns(clean_df)
for key in dataset_dict.keys():

    alk_vs_cols(clean_df, variables_list=dataset_dict[key])
    break

In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(40, 20),sharex=False, sharey=True)
axes = axes.flatten()
for i, col in enumerate(clean_df.columns.to_list()):
    y = clean_df.talk_gp
    x = clean_df[col]
    axes[i].scatter(x=x, y=y, alpha=0.5)
    axes[i].set_xlabel(str(col))
    
    from statsmodels.nonparametric.smoothers_lowess import lowess
    smoothed = lowess(endog=y, exog=x, frac=0.2, it=3, return_sorted=True)
    x_smooth, y_smooth = smoothed[:, 0], smoothed[:, 1]
    axes[i].plot(x_smooth, y_smooth, color='red', linewidth=2)

    # pcorr = pearson_corr['talk_gp'].loc[col]
    # taucorr = kendall_corr['talk_gp'].loc[col]
    # axes[i].text(0.05, 0.95, f"r= {pcorr:.2f}, tau = {taucorr:.2f}", transform=axes[i].transAxes,verticalalignment='top',bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
plt.tight_layout()
plt.show();



# Key Parameters:

# frac: Controls the smoothing.

# Small values → less smoothing (more sensitive to noise).
# Large values → more smoothing (less detail).


# it: Number of robustifying iterations to reduce the effect of outliers.
# return_sorted=True: Ensures output is sorted by x.


#  Edge Cases Handled:

# Works with unevenly spaced x values.
# Handles outliers better with it > 0.
# Robust to small datasets (but frac should be adjusted accordingly).



In [ ]:
# def histplots_1D(df):
    
#     columns = df.columns.to_list
#     col_length = len(columns)
    
    

fig, axes = plt.subplots(4, 8, figsize=(40, 20),sharex=False, sharey=False)
axes = axes.flatten()

for i, col in enumerate(clean_df.columns.to_list()):
    
    sns.histplot(clean_df[col],kde=True, ax= axes[i])
    axes[i].set_xlabel(str(col))
    if not pd.api.types.is_datetime64_any_dtype(clean_df[col]):
        axes[i].text(0.05, 0.95, f"Skewness: {clean_df[col].skew():.2f}", transform=axes[i].transAxes,verticalalignment='top',bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show();




In [ ]:
fig, axes = plt.subplots(7, 1, figsize=(40, 20),sharex=False, sharey=False)
axes = axes.flatten()

for i, col in enumerate(spatiotemporal_cols):

    if col == 'lat_gp' or col == 'lon_gp':
        min = df[col].min()
        max = df[col].max()
        sns.histplot(clean_df[col], binwidth=0.25, binrange=[min-1, max+1], kde=True, ax= axes[i])
    else:
        sns.histplot(clean_df[col],kde = True, ax= axes[i])
    axes[i].set_xlabel(col)

plt.tight_layout()
plt.show();


In [ ]:
# sns.violinplot(x='lat_gp', y='lon_gp', data=clean_df, inner=None, palette='Set2')
# sns.swarmplot(x='lat_gp', y='lon_gp', data=clean_df, color='k', size=2)
# plt.title('Combined Violin and Swarm Plot')
# plt.show()


# from matplotlib.colors import LogNorm

# plt.style.use('default')
# selected = ['talk_gp','salinity_gp','salt_soda','sss_glorys','sss_multiobs','sss_cciS']
# g = sns.pairplot(clean_df[selected], kind='hist')
# g.map_offdiag(sns.histplot, norm=LogNorm())
# plt.show()

In [ ]:
# def samples_map(df: pd.DataFrame, datasets_dict: munch.Munch, ds_name: str, varname: str):
    
#     fig, ax = blank_map()
    
#     sc = ax.scatter(
#     df[datasets_dict[ds_name].coord.lon], 
#     df[datasets_dict[ds_name].coord.lat],
#     c= df[varname],
#     s=20,
#     cmap="viridis",
#     transform=ccrs.PlateCarree()
#     )

#     plt.colorbar(sc, ax=ax, label=varname)

#     plt.show()

In [ ]:
# from statsmodels.tsa.stattools import acf

# def compute_acf(x, nlags=10):
    
#     acf_vals = acf(x, nlags=nlags, fft=False)
    
#     return pd.Series({
#         'lag1': acf_vals[1],
#         'lag2': acf_vals[2],
#         'lag3': acf_vals[3],
#         'lag4': acf_vals[4],
#         'lag5': acf_vals[5],
#         'lag6': acf_vals[6],
#         'lag7': acf_vals[7],
#         'lag8': acf_vals[8],
#         'lag9': acf_vals[9],
#         'lag10': acf_vals[10]
#     })

# acf_df = (
#     df.sort_values(['expocode_gp', 'time_gp'])
#       .groupby('expocode_gp')['talk_gp']
#       .apply(lambda x: compute_acf(x, 10))
#       .reset_index()
# )

# acf_df.info()


In [ ]:
# regressions
# check for multicolinearity
# lags or autocorr within cruise?

TODO:
-regressions
-check for multicolinearity
-lags or autocorr within cruise?

In [ ]:
import numpy as np
import pandas as pd

series = np.linspace(1, 100)

def kth_autocorrelation(series, max_k=10):
    results = []

    for k in range(1, max_k + 1):
        x1 = series[:-k]
        x2 = series[k:]
        corr = np.corrcoef(x1, x2)[0, 1]
        results.append(corr)

    # Create DataFrame with one row and columns as lag1, lag2, ...
    corr_df = pd.DataFrame({f'lag{y}': [results[y-1]] for y in range(1, max_k + 1)})
    return corr_df


group_corr = df.groupby('expocode_gp')['talk_gp'].apply(lambda x: kth_autocorrelation(x, max_k=3))

print(group_corr)
# # Example: df columns = ['group', 'k1', 'k2', 'k3']
# x = df['expocode_gp']           # categorical x-axis
# k_columns = [col for col in df.columns if col != 'expocode_gp']

# plt.figure(figsize=(20, 10))

# for col in k_columns:
#     plt.plot(x, df[col], marker='o', label=col)  # one line per correlation

# plt.xlabel('Group')
# plt.ylabel('Correlation value')
# plt.title('Correlations per group')
# plt.legend()
# plt.grid(True)
# plt.xticks(rotation=45)   # rotate group names for readability
# plt.tight_layout()        # adjust spacing
# plt.show()



In [ ]:
kernel_density_plot(clean_df,'Spatiotemporal samples distribution GLODAP',spatiotemporal_cols)

In [ ]:
# # statistics
# df.describe()[cnames[1:21]].T


# # missing values in each columns
# df.isnull().sum()
# #how many features per column are unique
# df.nunique()

In [ ]:
kernel_density_plot(clean_df, variables_list= salinity_cols)

# A simple ML regression (Geeks4Geeks)

In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler, OneHotEncoder
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score


# # Load dataset
# df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

# # Select relevant features
# features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
# df = df[features + ['Survived']].dropna()  # Drop rows with missing values

# # Display the first few rows
# print(df.head())

# # Load dataset
# df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

# # Select relevant features
# features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
# df = df[features + ['Survived']].dropna()  # Drop rows with missing values

# # Display the first few rows
# print(df.head())

# # Define target and features
# X = df[features]
# y = df['Survived']

# # Split into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Display the shape of the data
# print(f"Training set shape: {X_train.shape}")
# print(f"Testing set shape: {X_test.shape}")

# # Define the pipeline
# pipeline = Pipeline([
#     ('preprocessor', preprocessor),  # Data transformation
#     ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))  # ML model
# ])

# # Train the model
# pipeline.fit(X_train, y_train)
# print("Model training complete!")

# # Make predictions
# y_pred = pipeline.predict(X_test)

# # Compute accuracy
# accuracy = accuracy_score(y_test, y_pred)
# print(f"Model Accuracy: {accuracy:.2f}")

# import joblib

# # Save the trained pipeline
# joblib.dump(pipeline, 'ml_pipeline.pkl')

# # Load the model
# loaded_pipeline = joblib.load('ml_pipeline.pkl')

# # Predict using the loaded model
# sample_data = pd.DataFrame([{'Pclass': 3, 'Sex': 'male', 'Age': 25, 'SibSp': 0, 'Parch': 0, 'Fare': 7.5, 'Embarked': 'S'}])
# prediction = loaded_pipeline.predict(sample_data)
# print(f"Prediction: {'Survived' if prediction[0] == 1 else 'Did not Survive'}")

# Train-test split

In [ ]:
# histogram(modulated_train_glodap_fisrt['lon_grp'], modulated_train_glodap_fisrt['lat_grp'], title = 'Train split modulated random')
# histogram(train_glodap_fisrt['lon_grp'], train_glodap_fisrt['lat_grp'], title = 'Train split random')
# histogram(glodap_first['lon_grp'], glodap_first['lat_grp'], title = 'Train split')

def train_test_map(train_df, test_df,lon_res=1, lat_res=1):
    
    plt.show();
    color_list = ['orange','blue']
    labels = ['train','test']
    
    for month in sorted(train_df.bins_month_gp.unique()):
        
        plot_dict = dict()
        
        
        
        plot_dict['train'] = {
            'dataframe' : train_df[train_df.bins_month_gp == month],
            'color' : 'purple',
        }
        
        
        plot_dict['test'] = {
            'dataframe' : test_df[test_df.bins_month_gp == month],
            'color' : 'orange',
        }


        fig, ax = blank_map(lon_res=lon_res, lat_res=lat_res)

        for key in plot_dict:
            
            ax.scatter(
                plot_dict[key]['dataframe']['lon_gp'],
                plot_dict[key]['dataframe']['lat_gp'],
                marker = 'x',
                s = 10,
                color = plot_dict[key]['color'],
                alpha= 0.5,
                transform = ccrs.PlateCarree(),
                label = str(key)
            )
        plt.legend()
        plt.title(month)
        plt.show()

def train_test_histogram(df, train_df, title = '', color = 'purple', label = 'train'):

    fig, axes = plt.subplots(6,1, figsize = (20,42))

    sns.histplot(df['lon_grp'], discrete = True, ax = axes[0], label = 'glodap')
    sns.histplot(train_df['lon_grp'], discrete = True, ax = axes[0], color= color,  label = label)
    #sns.histplot(modulated_train_glodap_fisrt['lon_grp'], discrete = True, ax = axes[0], color= 'red',  label = 'smoothed')

    axes[0].legend()
    axes[0].set_xlabel('lon (4deg bins)')

    sns.histplot(df['lat_grp'], discrete = True, ax = axes[1], label = 'glodap')
    sns.histplot(train_df['lat_grp'], discrete = True, ax = axes[1], color= color, label = label)
    #sns.histplot(modulated_train_glodap_fisrt['lat_grp'], discrete = True, ax = axes[1], color= 'red',label = 'smoothed')
    axes[1].legend()
    axes[1].set_xlabel('lat (2def bins)')


    sns.histplot(df['bins_year_gp'], discrete = True, ax = axes[2], label = 'glodap')
    sns.histplot(train_df['bins_year_gp'], discrete = True, ax = axes[2], color= color,  label = label)
    #sns.histplot(modulated_train_glodap_fisrt['bins_year_gp'], discrete = True, ax = axes[2], color= 'red',  label = 'smoothed')

    axes[2].legend()
    axes[2].set_xlabel('year')


    sns.histplot(df['bins_month_gp'], discrete = True, ax = axes[3], label = 'glodap')
    sns.histplot(train_df['bins_month_gp'], discrete = True, ax = axes[3], color= color,  label = label)
    #sns.histplot(modulated_train_glodap_fisrt['bins_month_gp'], discrete = True, ax = axes[3], color= 'red',  label = 'smoothed')

    axes[3].legend()
    axes[3].set_xlabel('month')

    sns.histplot(df['bins_8D_gp'], discrete = True, ax = axes[4], label = 'glodap')
    sns.histplot(train_df['bins_8D_gp'], discrete = True, ax = axes[4], color= color,  label = label)
    #sns.histplot(modulated_train_glodap_fisrt['bins_8D_gp'], discrete = True, ax = axes[4], color= 'red',  label = 'smoothed')

    axes[4].legend()
    axes[4].set_xlabel('8D bins')

    sns.countplot(x=df['bins_salinity_gp'], order=df['bins_salinity_gp'].cat.categories, ax=axes[5], label = 'glodap')
    sns.countplot(x=train_df['bins_salinity_gp'], order=df['bins_salinity_gp'].cat.categories, ax= axes[5], color = color, label = label)
    axes[5].legend()
    axes[5].set_xlabel('salinity bins')

    plt.suptitle(title)
    plt.tight_layout()
    plt.show();


In [ ]:
import xarray as xr
ds = xr.open_dataset("woa23_decav_s00_01.nc", decode_times=False)
ds = ds.where(ds.depth == 0.0, drop=True)
ds_vars = list(ds.data_vars)
ds_vars.remove('s_an')
ds_vars = ds_vars + ['depth','time']
ds = ds.drop_vars(ds_vars)
ds = ds.isel(time=0, depth=0)
ds

# salinity masks
sal32 = ds.where(ds.s_an < 32)
sal32_34 = ds.where((ds.s_an >= 32) & (ds.s_an < 34))
sal34_36 = ds.where((ds.s_an >= 34) & (ds.s_an < 36))
sal36 = ds.where(ds.s_an >= 36)

In [ ]:
s = ds['s_an'].squeeze()

sal_class = xr.full_like(s, np.nan)

sal_class = xr.where(s < 32, 1, sal_class)
sal_class = xr.where((s >= 32) & (s < 34), 2, sal_class)
sal_class = xr.where((s >= 34) & (s < 36), 3, sal_class)
sal_class = xr.where(s >= 36, 4, sal_class)

In [ ]:


plt.show();

In [ ]:
def salinity_regions(df, salinity_class_ds):


    # # convert longitude if needed (-180..180 -> 0..360)
    # df["lon"] = df["lon"] % 360

    # --- 4. Convert dataframe coordinates to xarray ---
    points = xr.Dataset(
        coords=dict(
            lat=("points", df["lat_gp"].values),
            lon=("points", df["lon_gp"].values),
        )
    )

    # --- 5. Sample salinity mask at point locations ---
    point_classes = sal_class.interp(
        lat=points.lat,
        lon=points.lon,
        method="nearest"
    )
    
    # --- 6. Attach region labels to dataframe ---
    return point_classes.values



In [ ]:
df["salinity_region"] = salinity_regions(df,sal_class)

fig = plt.figure(figsize=(14,10))
ax = plt.axes(projection=ccrs.PlateCarree())

sal_class.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="Spectral_r",
    add_colorbar=True,
    alpha = 0.7
)

ax.scatter(
        x=df[df.salinity_region.isna()== True].lon_gp,
        y= df[df.salinity_region.isna()== True].lat_gp,
        marker = 's',
        s = 20,
        color = 'black',
        transform = ccrs.PlateCarree()
    )

ax.coastlines()

# Define tick locations
lon_ticks = np.arange(-180, 0, 10)
lat_ticks = np.arange(-90, 91, 10)

gl = ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=False,
    linewidth=0.5,
    color='gray',
    alpha=0.5,
    linestyle='--'
)

gl.xlocator = plt.FixedLocator(lon_ticks)
gl.ylocator = plt.FixedLocator(lat_ticks)
plt.show();


df['salinity_region'] = df['salinity_region'].fillna(0)

# nan_regions = df[df.salinity_region.isna()== True]
# nan_expocodes = nan_regions.expocode_gp.unique()
# for expocode in nan_expocodes:
#     mean_sal_region = round(df[df.expocode_gp == expocode].salinity_region.mean())
#     nan_regions[nan_regions.expocode_gp == expocode].salinity_region = mean_sal_region
# df[df.salinity_region.isna()== True] = nan_regions

# print(len(nan_regions.salinity_region))
# # nans_regions =

# nan_regions = nan_regions[(nan_regions.lat_gp > 30)&(nan_regions.lat_gp < 60)&(nan_regions.lon_gp > 0)&(nan_regions.lon_gp < 250)]

# print(nan_regions.expocode_gp.unique())

# min_lat = nan_regions[nan_regions.expocode_gp == '49NZ20210713'].lat_gp.min()-20
# max_lat = nan_regions[nan_regions.expocode_gp == '49NZ20210713'].lat_gp.max()+20
# # min_lon = nan_regions[nan_regions.expocode_gp == '49NZ20210713'].lon_gp.min()
# # max_lon = nan_regions[nan_regions.expocode_gp == '49NZ20210713'].lon_gp.max()

fig = plt.figure(figsize=(14,10))
ax = plt.axes(projection=ccrs.PlateCarree())

sal_class.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="Spectral_r",
    add_colorbar=True,
    alpha = 0.7
)

ax.scatter(
        x=df[df.salinity_region.isna()== True].lon_gp,
        y= df[df.salinity_region.isna()== True].lat_gp,
        marker = 's',
        s = 20,
        color = 'red',
        transform = ccrs.PlateCarree()
    )

ax.coastlines()

# Define tick locations
lon_ticks = np.arange(-180, 0, 10)
lat_ticks = np.arange(-90, 91, 10)

gl = ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=False,
    linewidth=0.5,
    color='gray',
    alpha=0.5,
    linestyle='--'
)

gl.xlocator = plt.FixedLocator(lon_ticks)
gl.ylocator = plt.FixedLocator(lat_ticks)
plt.show();

# print(nans_regions.info())

In [ ]:


print(df.size)

loss and accuracy curves to detect overfitting
residual vs predictors plots
accuracy/loss vs number of trees
check variance(target_normalized) vs driver, if you see small driver → very large variance, heteroscedasticity is strong
might be better: Instead of dividing by the driver, use logs (transform multiplicative relation in additive one)
do experiments with sTA and TA predictions to see the gain in accuracy, esp when working with noisy salinity datasets. Or maybe heteroscedasticity is good for the iterative experiments if it amplify the noise on small salinity values/big measurment errors but not sure

In [ ]:
print(nan_regions[nan_regions.expocode_gp == '49NZ20210713'].lat_gp)
print(nan_regions[nan_regions.expocode_gp == '49NZ20210713'].lon_gp)

In [ ]:
plt.show()
fig = plt.figure(figsize=(14,10))
ax = plt.axes(projection=ccrs.PlateCarree())

# sal_class.plot(
#     ax=ax,
#     transform=ccrs.PlateCarree(),
#     cmap="Spectral_r",
#     add_colorbar=True
# )

ax.scatter(
        x=df.lon_gp,
        y= df.lat_gp,
        c= df['salinity_region'],
        marker = 'x',
        s = 10,
        cmap = "Spectral_r",
        transform = ccrs.PlateCarree()
    )

ax.coastlines()

plt.show();

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
# sss.get_n_splits()

train_index, test_index = sss.split(
    X=df.drop(columns='talk_gp'), 
    y=df['salinity_region'])

train_df = clean_df[train_index]
test_df = clean_df[test_index]

train_test_map(train_df, test_df)


In [ ]:
plt.show()
fig, ax = blank_map()
lon = sal32_34.lon
lat = sal32_34.lat
var = sal32_34.s_an
ax.contourf(sal_class, transform=ccrs.PlateCarree(), extend="both", cmap='Spectral_r')

plt.show();

In [ ]:
def smoothed_random_split(df, percentage_test=0.2, alpha=1.5):
    
    N_total = len(df)
    N_test_total = int(N_total * percentage_test)

    # Compute group sizes
    group_sizes = df.groupby(['lat_grp','lon_grp','bins_month_gp'], observed=True).size()

    # Smoothed weights: flatten large bins
    weights = group_sizes ** alpha
    weights = weights / weights.sum()

    # Allocate test counts per group
    n_per_group = (weights * N_test_total).round().astype(int)

    # Fix rounding drift
    diff = N_test_total - n_per_group.sum()
    if diff != 0:
        adjust_idx = n_per_group.sort_values(ascending=False).index[:abs(diff)]
        n_per_group.loc[adjust_idx] += np.sign(diff)

    # Convert to dict: tuple -> int
    n_per_group_dict = n_per_group.to_dict()

    # Sample per group
    test_df = (
    df.groupby(['lat_grp','lon_grp','bins_salinity_gp'], group_keys=False, observed = True)
        .apply(lambda g: g.sample(
            n=min(n_per_group_dict[g.name], len(g)),
            random_state=42
        ))
    )

    # Remaining rows go to train
    train_df = df.drop(test_df.index)

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


def random_binned_split(df, percentage_test=0.2):
    
    train_df = df.copy(deep=True)

    # Sample independently inside each bin
    test_df = (
        df
        .groupby(['lat_grp', 'lon_grp','bins_salinity_gp'], group_keys=False, observed = True)
        .sample(frac=percentage_test, random_state=42)
    )

    # Everything else becomes train
    train_df = train_df.drop(test_df.index)

    # # Optional: reset indices
    # train_df = train_df.reset_index(drop=True)
    # test_df = test_df.reset_index(drop=True)
    
    return train_df, test_df

def random_split(df, percentage_test=0.2, plot = False):
    
    test_df = df.sample(frac = percentage_test, random_state=42)
        # Sample independently inside each bin
    test_df = (
        df
        .groupby(['expocode_gp'], group_keys=False, observed = True)
        .sample(frac=percentage_test, random_state=42)
    )
    train_df = df.copy(deep=True).drop(test_df.index)
    
    if plot == True:
        train_test_map(train_df, test_df, lon_res=1, lat_res=1)
        train_test_histogram(df, train_df, title = 'Train vs. GLODAP')
        train_test_histogram(df, test_df, title = 'Test vs. GLODAP', label='test', color = 'orange')
    
    return train_df, test_df


# for group in glodap_first.groupby(['lat_grp','lon_grp']):
#     group.reset_index()
    

In [ ]:
x_multi = [clean_df['lat_grp'], train_df['lat_grp'], test_df['lat_grp']]
plt.hist(x_multi, histtype='bar')
plt.show()

In [ ]:
lon_res = 2
lat_res = 1

lon_grp, lat_grp = spatial_bins(clean_df,lon_mesh=lon_res,lat_mesh=lat_res, histogram=True)

clean_df['lat_grp'] = lat_grp
clean_df['lon_grp'] = lon_grp

#salinity_global_quantiles
#quantiles = (np.array([0, 1, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 99, 100]) / 100)
quantiles =(np.array([0, 1, 5, 25, 50, 75, 95, 99, 100])/100)
bins_sal, bins_midpoints_sal = salinity_bins(clean_df, quantiles=quantiles, plot=False)

clean_df['bins_mid_salinity_gp'] = bins_midpoints_sal
clean_df['bins_salinity_gp'] = bins_sal

train_df, test_df = random_split(clean_df, percentage_test=0.2, plot = True)

large spatiotemporal bins
global salinity bins 
groupby on sal-spatial bins
check that all years are represented a correct amount

In [ ]:
lon_res = 20
lat_res = 16

lon_grp, lat_grp = spatial_bins(clean_df,lon_mesh=lon_res,lat_mesh=lat_res, histogram=True)

clean_df['lat_grp'] = lat_grp
clean_df['lon_grp'] = lon_grp

#salinity_global_quantiles
#quantiles = (np.array([0, 1, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 99, 100]) / 100)
quantiles =(np.array([0, 1, 5, 25, 50, 75, 95, 99, 100])/100)
bins_sal, bins_midpoints_sal = salinity_bins(clean_df, quantiles=quantiles, plot=False)
clean_df.insert(14,'bins_salinity_gp', bins_sal)
clean_df.insert(15, 'bins_mid_salinity_gp',bins_midpoints_sal)


train_glodap, test_glodap = random_split(clean_df, percentage_test=0.4)

train_test_histogram(clean_df, train_glodap)
train_test_map(train_glodap, test_glodap,lon_res=lon_res, lat_res= lat_res)

# train_glodap_first.describe()
# test_glodap_first.describe()
# clean_df.describe()

In [ ]:
train_test_map(train_glodap, test_glodap, lat_res=lat_res, lon_res=lon_res)

In [ ]:
glodap_first = clean_df.groupby(['expocode_gp', 'lat_gp', 'lon_gp']).first().reset_index()

print(glodap_first.depth_gp.unique())
print(clean_df.depth_gp.unique())


sns.histplot(clean_df.depth_gp, color= 'orange')
sns.histplot(glodap_first.depth_gp)
plt.show()

In [ ]:
from glob import glob


lat_res = 1
lon_res = 1

glodap_first = clean_df.groupby(['expocode_gp', 'lat_gp', 'lon_gp']).first().reset_index()

lon_grp, lat_grp = spatial_bins(glodap_first,lon_mesh=lon_res,lat_mesh=lat_res, histogram=False)

glodap_first['lat_grp'] = lat_grp
glodap_first['lon_grp'] = lon_grp

train_glodap_first, test_glodap_first = random_split(glodap_first, percentage_test=0.2)

# train_test_histogram(glodap_first,train_glodap_first)
# train_test_map(train_glodap_first, test_glodap_first, lat_res=lat_res, lon_res=lon_res)

In [ ]:
train_test_map(train_glodap_first, test_glodap_first, lat_res = lat_res, lon_res = lon_res)

In [ ]:
# samples_map(test_glodap_first[test_glodap_first['bins_month_gp']==1])
# samples_map(train_glodap_fisrt[train_glodap_fisrt['bins_month_gp']==1],'color = orange')


In [ ]:
samples_map(glodap_first, source_dataset='grp', grid_res=2)



# Inference datasets

In [ ]:
from functools import lru_cache
import xarray as xr




URL_FOR_ANYWHERE_ACCESS = "https://data.up.ethz.ch/shared/OceanSODA-ETHZv2/inference_for_gregor2024/data_8daily_25km_v01.zarr/"
FNAME_FOR_SEA_ACCESS = "/net/sea/work/gregorl/projects/OceanSODA-ETHZv2/data-in/inference/data_8daily_25km_v01.zarr/"




@lru_cache(maxsize=1)
def open_gridded_dataset(fname = FNAME_FOR_SEA_ACCESS)->xr.DataTree:
   return xr.open_datatree(fname, chunks="auto", engine='zarr')




@lru_cache(maxsize=10)
def open_gridded_feature(variable: str)-> xr.DataArray:
   ds_tree = open_gridded_dataset()


   ds_sample = ds_tree['2015']
   nice_list_vars = "\n\t".join(list(ds_sample.data_vars))
   assert variable in ds_sample, f"Variable '{variable}' not found in dataset. Available variables: [\n\t{nice_list_vars}\n]"


   da_list = [ds_tree[key][variable] for key in ds_tree.keys() if variable in ds_tree[key]]
   da = xr.concat(da_list, dim="time")


   return da


In [ ]:
data_tree = open_gridded_dataset()

inference_da = data_tree.get('2024')

In [ ]:
inference_da